# Itemset — native Julia on Colab

Select **Runtime → Change runtime type → Julia**. All code cells run directly in the Julia kernel. Use Julia 1.12+ within 1.x; if needed select runtime version **2026.07** (Julia 1.12.6) or a newer compatible runtime. See [Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html).

For this local repository, export a committed ZIP with `git archive --format=zip --output=../Itemset-colab.zip HEAD`, upload it through Colab's **Files** sidebar to `/content/Itemset-colab.zip`, and run all cells. Alternatively set `REPO_URL` to `https://github.com/NgTHung/Itemset` and `REVISION` to an exact commit below. Use Julia 1.13.2 to reproduce the report's codebook; compatible minor versions may produce different seeded encodings.

The notebook verifies the environment and tests, then runs the Semeion image pilot. Keep the ZIP revision with the output.

In [ ]:
REPO_URL = ""  # Optional future public GitHub HTTPS URL.
REVISION = ""  # Exact commit required for cloning; record the ZIP's commit too.
ROOT = get(ENV, "ITEMSET_ROOT", isdir("/content") ? "/content/Itemset" : pwd())
ARCHIVE = "/content/Itemset-colab.zip"  # Upload through Colab's Files sidebar.

if !isempty(REPO_URL)
    isempty(REVISION) && error("Set an exact commit in REVISION for reproducibility")
    ispath(joinpath(ROOT, ".git")) || run(`git clone $REPO_URL $ROOT`)
    run(`git -C $ROOT checkout --detach $REVISION`)
elseif !isfile(joinpath(ROOT, "Project.toml"))
    isfile(ARCHIVE) || error("Upload Itemset-colab.zip through the Files sidebar first")
    mkpath(ROOT)
    run(`unzip -q -o $ARCHIVE -d $ROOT`)
end

isfile(joinpath(ROOT, "Project.toml")) || error("Repository root must contain Project.toml")
ROOT = abspath(ROOT)
cd(ROOT)
println("Repository: ", ROOT)
println("Revision: ", ispath(".git") ? readchomp(`git rev-parse HEAD`) :
        isempty(REVISION) ? "local archive; record its commit before final experiments" : REVISION)


In [ ]:
using Pkg
v"1.12" <= VERSION < v"2" ||
    error("Select a Colab Julia 1.12+ runtime (e.g. runtime version 2026.07)")
Pkg.activate(ROOT)
Pkg.instantiate()

using ItemsetMining
using Random
println("Julia: ", VERSION, " | threads: ", Threads.nthreads(), " | seed: ", DEFAULT_SEED)
println("Project: ", Base.active_project())
artifact_manifest = get(ENV, "ITEMSET_ARTIFACT_MANIFEST", "")
notebook_artifacts = Dict{String,String}()


In [ ]:
# Execute setup and tests directly inside this Julia kernel.
# Separate modules keep test definitions out of the notebook workspace.
Base.include(Module(:ItemsetSetupCheck), joinpath(ROOT, "scripts", "check_setup.jl"))
module ItemsetSetupTests
    include(joinpath(Main.ROOT, "test", "runtests.jl"))
end


In [ ]:
transactions = read_transactions(joinpath(ROOT, "data", "toy", "baskets.txt"))
minsup_count = support_threshold(0.5, length(transactions))
(; transactions, minsup_count)


## Hung — Eclat, tests, scalability, output size

`mine` with `variant=:baseline` runs the paper-based Eclat described in `docs/eclat.md`. The cell below mines the toy baskets at minsup 0.5 and prints the SPMF output, which matches `data/toy/baskets.expected.txt`. The test cell above checks the baseline against every toy fixture and a brute-force count.

In [ ]:
patterns = mine(transactions; minsup=0.5, variant=:baseline)
write_itemsets(stdout, patterns)

## Thien — Benchmarks, market baskets, runtime, transaction width

The next cell downloads the benchmark datasets, checks each file's SHA-256 against `data/benchmark/manifest.toml`, and prints the statistics computed from the files. Timing runs use `experiments/run.jl` as described in `experiments/README.md`. SPMF series need Java 25 or later.

The cell after it downloads Online Retail II, checks its SHA-256 against `data/raw/manifest.toml`, cleans it into one basket per invoice, and writes the baskets and the stock-code mapping to `data/raw/market_basket/`. `applications/market_basket/README.md` explains each cleaning step.

The cell after that mines the baskets with the team's Eclat, derives association rules, and writes the top-10 rules, the top-10 product groups, and the threshold, per-year, and large-basket checks to `results/raw/market_basket/`; it takes about three minutes. `docs/market-basket-rules-2026-10-29.md` explains the tables. The runtime cell prints the medians, quartiles, and ratios of the BBQ cohort and shows the report's runtime figure. The width cell regenerates the synthetic databases and shows the report's width figure. The last cell profiles the stages of the optimized CLI path (read, mine, write, and the share of each part) in about a minute. The figures come from `docs/report/figures/`, which `scripts/plot_runtime_curves.py` and `scripts/plot_width_curves.py` write, because the plots use matplotlib and this kernel is Julia.

Provenance: Julia 1.13.2 with one thread, SPMF 2.66, and `DEFAULT_SEED` (42) for every stochastic step. The runtime cohort is `bbq-all-variants-cli-5repeat-20261117T041922Z`, described in `docs/runtime-curves-2026-11-17.md` with its checksums and commands. The width experiment is described in `docs/width-runtime-2026-11-11.md`, and the stage profile in `docs/future-optimizations-2026-11-22.md`.

In [ ]:
# Prepare and verify the benchmarks in data/benchmark/.
module PrepareBenchmarks
    include(joinpath(Main.ROOT, "experiments", "prepare_benchmarks.jl"))
end
PrepareBenchmarks.main(String[])


In [ ]:
# Clean Online Retail II into invoice baskets (about 95 MB download).
module PrepareBaskets
    include(joinpath(Main.ROOT, "applications", "market_basket", "prepare.jl"))
end
PrepareBaskets.main(String[])
items = split.(readlines(joinpath(ROOT, "data", "raw", "market_basket", "items.tsv"))[2:end], '\t')
first(sort(items; by=row -> -parse(Int, row[4])), 10)


In [ ]:
# Mine the baskets, derive association rules, and check them (about three minutes).
module MarketBasketRules
    include(joinpath(Main.ROOT, "applications", "market_basket", "analyze.jl"))
end
MarketBasketRules.main(String[])
notebook_artifacts["market_basket"] = joinpath(ROOT, "results", "raw", "market_basket")
table(name) = split.(readlines(joinpath(ROOT, "results", "raw", "market_basket", name)), '\t')
table("top_groups.tsv")


In [ ]:
# Show a figure of the report, or name its path when the kernel cannot display images.
function show_figure(path)
    isfile(path) || return println("Figure not found: ", path)
    try
        display("image/png", read(path))
    catch
        println("Figure: ", path)
    end
end

In [ ]:
# Runtime against minsup: median and quartiles from a verified benchmark run (DAT-19).
# The default folder is the BBQ cohort in data/raw/. Set ITEMSET_RUNTIME_RUN to use another run folder
# that holds runs.csv, such as one from experiments/run.jl.
module RuntimeCurves
    include(joinpath(Main.ROOT, "scripts", "runtime_curves.jl"))
end
run_dir = get(ENV, "ITEMSET_RUNTIME_RUN",
              joinpath(ROOT, "data", "raw", "bbq-all-variants-cli-5repeat-20261117T041922Z"))
if !isfile(joinpath(run_dir, "runs.csv"))
    println("No runs.csv in ", run_dir, ". Put the BBQ run folder there or set ITEMSET_RUNTIME_RUN.")
else
    RuntimeCurves.main(["--out", joinpath(ROOT, "results", "figures", "runtime"),
        "--ratio", "Baseline/Optimized", "--ratio", "Optimized/PrePost+",
        "Baseline=$run_dir:baseline", "Optimized=$run_dir:optimized", "SPMF Eclat=$run_dir:spmf",
        "PrePost+=$run_dir:prepost"])
end
show_figure(joinpath(ROOT, "docs", "report", "figures", "runtime-curves.png"))

In [ ]:
# Transaction width against Eclat runtime (DAT-22): regenerate the eight synthetic databases from the
# frozen manifest and print the density, output volume, and matched support count of each (about 10 seconds).
# The timed runs are in docs/width-runtime-2026-11-11.md and use experiments/configs/width-*.toml.
module WidthRun
    include(joinpath(Main.ROOT, "scripts", "width_experiment.jl"))
end
WidthRun.main(["pilot", "--out", joinpath(ROOT, "results", "figures", "width")])
show_figure(joinpath(ROOT, "docs", "report", "figures", "width-curves.png"))

In [ ]:
# Stage profile of the optimized CLI path (DAT-23): the time of the read, mine, and write stages and the share of
# each part of the search, the root layout, and the output at eleven points (about a minute). It needs the
# benchmarks that the first cell of this section prepares. The method and the results are in
# docs/future-optimizations-2026-11-22.md.
module StageProfileRun
    include(joinpath(Main.ROOT, "scripts", "stage_profile.jl"))
end
StageProfileRun.main(["--out", joinpath(ROOT, "results", "figures", "stage-profile"), "--repeats", "3",
                      joinpath(ROOT, "experiments", "configs", "stage-profile.toml")])

## Phat: deterministic Semeion encoding

The [UCI Semeion dataset](https://archive.ics.uci.edu/dataset/178/semeion+handwritten+digit) contains binary 16×16 handwritten digits under CC BY 4.0. The preprocessing cell verifies the source, fits a private seed-42 patch codebook, and keeps one distinct-word transaction per image, including empty cases. `manifest.toml` records settings, revision, source and artifact hashes, width distribution and density. SVGs show observed patches assigned to their words. The whole dataset fits this descriptive codebook; this is not a generalization evaluation. Position and multiplicity are lost in the word sets. The following pilot cell mines the same encoding at three thresholds.

In [ ]:
semeion_path = joinpath(ROOT, "data", "raw", "semeion", "semeion.data")
pilot_output = joinpath(mktempdir(; cleanup=isempty(artifact_manifest)), "semeion-pilot")
module ImagePilotRun
    include(joinpath(Main.ROOT, "scripts", "image_pilot.jl"))
end
pilot = ImagePilotRun.run_mining_command([semeion_path, pilot_output])
notebook_artifacts["encoding"] = pilot_output
notebook_artifacts["semeion"] = pilot_output
encoding_manifest = ImagePilotRun.SemeionPilot.TOML.parsefile(joinpath(pilot_output, "manifest.toml"))
display(encoding_manifest["statistics"])
display(HTML(read(joinpath(pilot_output, "digits.svg"), String)))
display(HTML(read(joinpath(pilot_output, "visual_words.svg"), String)))


In [ ]:
println(read(joinpath(pilot_output, "selected_patterns.tsv"), String))
display(HTML(read(joinpath(pilot_output, "supporting-0.2.svg"), String)))
display(HTML(read(joinpath(pilot_output, "supporting-0.05.svg"), String)))


### Analyze patterns and construct features

Use one encoding for both the patterns and their image IDs. The table keeps each digit's full denominator, including empty transactions. Counts describe this fitted dataset. A pattern query retrieves images containing all its words; it does not retain patch positions.

For a fixed pattern list `P`, feature `z[i,j]` is 1 exactly when `P[j]` is a subset of image transaction `T[i]`. The hand-worked example below yields `[1,1,0]` and an empty image yields zeros. For a predictive evaluation, split images first, fit the codebook and mine patterns only on training images, and freeze feature column order for test images. The all-data descriptive run below does not measure held-out classification accuracy.


In [ ]:
analysis_encoding = pilot.encoding
analysis_runs = pilot.runs
analysis_patterns = ImagePilotRun.SemeionPilot.select_patterns(last(analysis_runs).patterns)
for pattern in analysis_patterns
    rows = ImagePilotRun.SemeionPilot.pattern_label_summary(analysis_encoding.images, analysis_encoding.transactions, pattern)
    @assert sum(row.support for row in rows) == last(analysis_runs).patterns[pattern]
    println("Pattern ", pattern, " | label | total | support | fraction | source IDs")
    display(rows)
end
feature_columns = sort!(collect(keys(last(analysis_runs).patterns)))
image_features = ImagePilotRun.SemeionPilot.itemset_features(analysis_encoding.transactions, feature_columns)
@assert vec(sum(image_features; dims=1)) == [last(analysis_runs).patterns[p] for p in feature_columns]
println("Image-feature matrix: ", size(image_features), " | fixed columns: ", feature_columns)
example_features = ImagePilotRun.SemeionPilot.itemset_features([[8, 13], Int[]], [(13,), (8, 13), (4, 8, 13)])
@assert example_features == Bool[1 1 0; 0 0 0]
display(example_features)


### Phat · Exact-map comparison and peak process memory

Put the SPMF v2.66 JAR at `tools/spmf/spmf.jar` using `tools/spmf/README.md`. Make Java 25 available on PATH or set `ITEMSET_JAVA` to its executable. The cell checks Chess at 90% minsup against the same SPMF Eclat output for both Julia variants. Expect two exact comparisons and two process peak RSS values in bytes. This single-point demo checks the setup. The full memory series uses five repeats at one medium threshold per dataset.

Run these commands from the repository root for the full experiments. Keep `runs.csv`, `environment.toml`, `references/` and the per-attempt `comparison.toml` files. Peak RSS includes startup, input, mining and output in the child process.

```sh
julia --startup-file=no --project=. experiments/run.jl experiments/configs/correctness.toml
julia --startup-file=no --project=. experiments/run.jl experiments/configs/peak-memory.toml
julia --startup-file=no --project=. scripts/mine_image_patterns.jl data/raw/semeion/semeion.data results/raw/semeion-k24 --vocabulary-size 24 --minsup 0.2,0.1,0.06,0.055,0.05
```

For vocabulary sensitivity, repeat the last command with K=16 and K=32 in separate output directories. Compare widths, empty images and pattern counts. Word IDs refer to the codebook fitted in each run.

In [ ]:
using TOML, SHA
java = get(ENV, "ITEMSET_JAVA", "java")
jar = joinpath(ROOT, "tools", "spmf", "spmf.jar")
if isfile(jar) && (isfile(java) || Sys.which(java) !== nothing)
    jar_sha = open(io -> bytes2hex(sha256(io)), jar)
    @assert jar_sha == "4aa468384693b2016b457ea310a08cc24aa83abd42d3169ad65b9027778dfe6a"
    smoke = TOML.parsefile(joinpath(ROOT, "experiments", "configs", "correctness.toml"))
    smoke["series"] = [Dict("dataset" => "chess", "minsup" => [0.9])]
    smoke_dir = mktempdir(; cleanup=isempty(artifact_manifest))
    config = joinpath(smoke_dir, "smoke.toml")
    open(io -> TOML.print(io, smoke), config, "w")
    runner = joinpath(ROOT, "experiments", "run.jl")
    output = joinpath(smoke_dir, "run")
    run(`$(Base.julia_cmd()) --startup-file=no --project=$ROOT $runner $config --out $output --java $java`)
    println("variant | exact | peak RSS (bytes)")
    attempts = readdir(joinpath(output, "attempts"); join=true)
    @assert length(attempts) == 2
    for attempt in attempts
        comparison = TOML.parsefile(joinpath(attempt, "comparison.toml"))
        measured = TOML.parsefile(joinpath(attempt, "measure.toml"))
        @assert comparison["matches"] && measured["peak_rss_bytes"] > 0
        println(basename(attempt), " | " , comparison["matches"], " | ", measured["peak_rss_bytes"])
    end
    notebook_artifacts["spmf-smoke"] = output
    println("Raw comparison and measurement files: ", output)
else
    println("SPMF check skipped. Follow the Java and JAR setup above to run it.")
end

if !isempty(artifact_manifest)
    open(io -> TOML.print(io, notebook_artifacts), artifact_manifest, "w")
end


## Clean-run checklist

Restart the Julia runtime and run all cells after modules are implemented. Record the Colab runtime version, Julia version, thread count, revision, dataset hashes, dependencies, parameters, stdout, and failures. Publish a public Colab link in README after verifying access and a complete clean run. A local run does not prove the hosted Colab runtime has been exercised.